#Import Libraries

In [1]:
!pip install evaluate jiwer -q

In [2]:
import torch
import numpy as np
from datasets import load_from_disk
from transformers import (
    Wav2Vec2Processor,
    Wav2Vec2ForCTC,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from dataclasses import dataclass
from typing import Dict, List, Union
import evaluate

#Load Dataset

In [3]:
# 1. Install dependencies
!pip install -q kagglehub datasets

# 2. Login Kaggle
import kagglehub
kagglehub.login()
# Masukkan token kamu saat diminta

# 3. Download dataset dari Kaggle
import os
from datasets import load_from_disk

dataset_path = kagglehub.dataset_download("nalaprogroup/data-nalapro-project03")
print("Downloaded to:", dataset_path)

# 4. Cek isi folder dulu
print("\nDaftar file:")
for root, dirs, files in os.walk(dataset_path):
    print(root)

# 5. Load dataset
dataset = load_from_disk(dataset_path)
print(" Dataset berhasil di-load:", dataset)

# 6. Ekstrak data
def prepare_data(split_data):
    audio_arrays, transcriptions, intent_classes = [], [], []
    for sample in split_data:
        audio_arrays.append(sample['audio']['array'])
        transcriptions.append(sample['transcription'])
        intent_classes.append(sample['intent_class'])
    return audio_arrays, transcriptions, intent_classes

train_audio, train_text, train_labels = prepare_data(dataset['train'])
val_audio,   val_text,   val_labels   = prepare_data(dataset['validation'])
test_audio,  test_text,  test_labels  = prepare_data(dataset['test'])

print(f"\nJumlah train  : {len(train_audio)}")
print(f"Jumlah val    : {len(val_audio)}")
print(f"Jumlah test   : {len(test_audio)}")
print(f"Sample rate   : {dataset['train'][0]['audio']['sampling_rate']} Hz")
print(f"Intent classes: {set(train_labels)}")

Using Colab cache for faster access to the 'data-nalapro-project03' dataset.
Downloaded to: /kaggle/input/data-nalapro-project03

Daftar file:
/kaggle/input/data-nalapro-project03
/kaggle/input/data-nalapro-project03/validation
/kaggle/input/data-nalapro-project03/test
/kaggle/input/data-nalapro-project03/train
 Dataset berhasil di-load: DatasetDict({
    train: Dataset({
        features: ['audio', 'intent_class', 'transcription'],
        num_rows: 1800
    })
    validation: Dataset({
        features: ['audio', 'transcription', 'intent_class'],
        num_rows: 56
    })
    test: Dataset({
        features: ['audio', 'transcription', 'intent_class'],
        num_rows: 57
    })
})

Jumlah train  : 1800
Jumlah val    : 56
Jumlah test   : 57
Sample rate   : 16000 Hz
Intent classes: {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13}
Kaggle credentials set.
Kaggle credentials successfully validated.


### Upercase

In [4]:
print("\n Converting text to uppercase...")

def uppercase_text(batch):
    batch["text"] = batch["transcription"].upper()
    return batch

import os
import tempfile

# Create a writable directory for datasets cache if it doesn't exist
# This ensures that temporary files can be written during the map operation
writable_cache_dir = "/tmp/datasets_cache"
os.makedirs(writable_cache_dir, exist_ok=True)

# Generate cache file names for each split in the dataset
# directing them to the writable temporary directory
cache_file_names = {
    split: os.path.join(writable_cache_dir, f"{split}_uppercase_text.arrow")
    for split in dataset.keys()
}

# Map the function, explicitly providing writable cache file paths
dataset = dataset.map(uppercase_text, load_from_cache_file=False, cache_file_names=cache_file_names)

print(f"   Sample: {dataset['train'][0]['text']}")


 Converting text to uppercase...


Map: 100%|##########| 1800/1800 [00:00<?, ? examples/s]

Map: 100%|##########| 56/56 [00:00<?, ? examples/s]

Map: 100%|##########| 57/57 [00:00<?, ? examples/s]

   Sample: BRAYON I RECENTLY AND I JUST WANT TO KNOW IF IT'S ALMOST OUT OF WORK I'M GOING NORTH AND GOING TO BE IN DIFFERENT COUNTRIES IN DIFFERENT COUNTRIES ONE IF IT'S A DIFFERENT WORD


#Load Model & Processor

In [5]:
print("\n Loading model...")
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
model = Wav2Vec2ForCTC.from_pretrained(
    "facebook/wav2vec2-base-960h",
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
)
model.freeze_feature_encoder()
for param in model.wav2vec2.encoder.layers[:6].parameters():
    param.requires_grad = False


 Loading model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-960h
Key                        | Status  | 
---------------------------+---------+-
wav2vec2.masked_spec_embed | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


#Preprocess

In [ ]:
'''import re

def clean_text(batch):
  text=batch["text"].lower()
  text=re.sub(r'[^a-z ]','',text)
  batch["text"]=text
  return batch

dataset = dataset.map(clean_text)'''

In [6]:
print("\n Preparing dataset for training...")

def prepare_dataset(batch):
    """Process audio (sudah 16kHz) dan text (sudah uppercase)"""
    audio = batch["audio"]

    # Audio sudah dalam 16kHz dari preprocessing sebelumnya
    inputs = processor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],  # = 16000
    )
    batch["input_values"] = inputs["input_values"][0]
  # label text
    labels = processor.tokenizer(batch["text"].lower())
    batch["labels"]=processor(text=batch["text"])["input_ids"]
    return batch


 Preparing dataset for training...


In [7]:
sample = prepare_dataset(dataset["train"][0])
print(sample)

{'audio': <datasets.features._torchcodec.AudioDecoder object at 0x7ba04c281790>, 'intent_class': 0, 'transcription': "brayon I recently and I just want to know if it's almost out of work I'm going north and going to be in different countries in different countries one if it's a different word", 'text': "BRAYON I RECENTLY AND I JUST WANT TO KNOW IF IT'S ALMOST OUT OF WORK I'M GOING NORTH AND GOING TO BE IN DIFFERENT COUNTRIES IN DIFFERENT COUNTRIES ONE IF IT'S A DIFFERENT WORD", 'input_values': array([ 0.00022872,  0.00022872,  0.00022872, ...,  0.00180551,
        0.00022872, -0.00089756], dtype=float32), 'labels': [24, 13, 7, 22, 8, 9, 4, 10, 4, 13, 5, 19, 5, 9, 6, 15, 22, 4, 7, 9, 14, 4, 10, 4, 29, 16, 12, 6, 4, 18, 7, 9, 6, 4, 6, 8, 4, 26, 9, 8, 18, 4, 10, 20, 4, 10, 6, 27, 12, 4, 7, 15, 17, 8, 12, 6, 4, 8, 16, 6, 4, 8, 20, 4, 18, 8, 13, 26, 4, 10, 27, 17, 4, 21, 8, 10, 9, 21, 4, 9, 8, 13, 6, 11, 4, 7, 9, 14, 4, 21, 8, 10, 9, 21, 4, 6, 8, 4, 24, 5, 4, 10, 9, 4, 14, 10, 20, 20, 5, 13

In [8]:
encoded_dataset = dataset.map(
    prepare_dataset,
    remove_columns=dataset["train"].column_names,
    num_proc=2,
)

In [9]:
sample = encoded_dataset["train"][0]
print("Decode:", processor.decode(sample["labels"]))
print("Original:", dataset["train"][0]["text"])

Decode: BRAYON I RECENTLY AND I JUST WANT TO KNOW IF IT'S ALMOST OUT OF WORK I'M GOING NORTH AND GOING TO BE IN DIFERENT COUNTRIES IN DIFERENT COUNTRIES ONE IF IT'S A DIFERENT WORD
Original: BRAYON I RECENTLY AND I JUST WANT TO KNOW IF IT'S ALMOST OUT OF WORK I'M GOING NORTH AND GOING TO BE IN DIFFERENT COUNTRIES IN DIFFERENT COUNTRIES ONE IF IT'S A DIFFERENT WORD


# Data Collator

In [10]:
sample = encoded_dataset["train"][0]

print(type(sample["input_values"]))
print(len(sample["input_values"]))
print(type(sample["input_values"][0]))
print(sample.keys())

<class 'list'>
682666
<class 'float'>
dict_keys(['input_values', 'labels'])


In [11]:
def simple_collator(features):
    input_values = [f["input_values"] for f in features]
    labels = [f["labels"] for f in features]

    batch = processor.pad(
        {"input_values": input_values},
        padding=True,
        return_tensors="pt"
    )

    labels_batch = processor.pad(
        labels={"input_ids": labels},
        padding=True,
        return_tensors="pt"
    )

    labels = labels_batch["input_ids"].masked_fill(
        labels_batch["attention_mask"].ne(1), -100
    )

    batch["labels"] = labels
    return batch

data_collator = simple_collator

In [12]:
batch = data_collator([encoded_dataset["train"][0]])
print(batch.keys())

KeysView({'input_values': tensor([[ 0.0002,  0.0002,  0.0002,  ...,  0.0018,  0.0002, -0.0009]]), 'labels': tensor([[24, 13,  7, 22,  8,  9,  4, 10,  4, 13,  5, 19,  5,  9,  6, 15, 22,  4,
          7,  9, 14,  4, 10,  4, 29, 16, 12,  6,  4, 18,  7,  9,  6,  4,  6,  8,
          4, 26,  9,  8, 18,  4, 10, 20,  4, 10,  6, 27, 12,  4,  7, 15, 17,  8,
         12,  6,  4,  8, 16,  6,  4,  8, 20,  4, 18,  8, 13, 26,  4, 10, 27, 17,
          4, 21,  8, 10,  9, 21,  4,  9,  8, 13,  6, 11,  4,  7,  9, 14,  4, 21,
          8, 10,  9, 21,  4,  6,  8,  4, 24,  5,  4, 10,  9,  4, 14, 10, 20, 20,
          5, 13,  5,  9,  6,  4, 19,  8, 16,  9,  6, 13, 10,  5, 12,  4, 10,  9,
          4, 14, 10, 20, 20,  5, 13,  5,  9,  6,  4, 19,  8, 16,  9,  6, 13, 10,
          5, 12,  4,  8,  9,  5,  4, 10, 20,  4, 10,  6, 27, 12,  4,  7,  4, 14,
         10, 20, 20,  5, 13,  5,  9,  6,  4, 18,  8, 13, 14]])})


#Metrics

In [16]:
wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = np.argmax(pred.predictions, axis=-1)

    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(label_ids, group_tokens=False)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

    print("PRED:", pred_str[:2])
    print("LABEL:", label_str[:2])

In [17]:
print(encoded_dataset["train"][0])

Output hidden; open in https://colab.research.google.com to view.

#Training

In [18]:
training_args = TrainingArguments(
    output_dir="./wav2vec2-minds14-base-960h",
    eval_strategy="steps",
    save_strategy="steps",
    eval_steps=50,
    save_steps=50,
    learning_rate=3e-5,
    per_device_train_batch_size=4,  # Reduced from 8 to 4
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    warmup_steps=0.1,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,

    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=5),
    ]
)

print("\n Starting training...")
trainer.train()


 Starting training...


Step,Training Loss,Validation Loss,Wer
50,0.000000,2.220504,0.406036


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

#Save Model

In [ ]:
print("\n Saving final model...")

model_save_path = "./wav2vec2-minds14-final"
trainer.save_model(model_save_path)
processor.save_pretrained(model_save_path)

print(f" Model saved to {model_save_path}")

In [ ]:
pred_ids = np.argmax(pred.predictions, axis=-1)
pred_str = processor.batch_decode(pred_ids)
label_str = processor.batch_decode(pred.label_ids, group_tokens=False)

print("PRED:", pred_str[0])
print("LABEL:", label_str[0])